# Assignment 2: Translation with a Small Transformer Model

In this assignment, you will build a small Transformer model to translate from French to English. This task follows <b>the Lab 4</b> but replaces the RNN-based model with a Transformer.

A Transformer is a neural network architecture that relies on self-attention mechanisms, and it is the basis for modern large language models. You can learn more about it in <b>the slides of Lecture 9, </b> and the paper [Attention Is All You Need](https://arxiv.org/abs/1706.03762).

**Submission instructions**: 
- Complete the code in the sections below, then rename this file as `<studentId>_<fullName>_assignment2.ipynb` and submit it to Moodle (e.g., `1234567_ShichaoMA_assignment2.ipynb`).
- Please keep all outputs shown in this Jupyter Notebook.

**Also fill in the followings**:
- Name:Wenjun Yu
- Student ID:25480677

<hr>

DO NOT MODIFY THIS SECTION

In [7]:
from __future__ import unicode_literals, print_function, division
from io import open
import math
import os
import random
import re
import time
import unicodedata

import torch
import torch.nn as nn
from torch import optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# In case you are using a Mac with M1/M2 chip
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if mps_available else "cpu"))
print(f"Using device: {device}")

Using device: cuda


In [8]:
if not os.path.isdir('data'):
    print("Downloading and unzipping data...")
    !wget https://download.pytorch.org/tutorial/data.zip
    !unzip data.zip
else:
    print("Data folder already exists. Skipping download.")

Data folder already exists. Skipping download.


<hr>

## Part A: Data Preparation 
**This part does not involve any task**

This part involves setting up the vocabulary, normalizing text, and creating a `DataLoader` for training.

- We need special tokens `PAD`, `SOS` (Start of Sentence), and `EOS` (End of Sentence) for sequence processing. `PAD` is used to make all sequences in a batch have the same length. `SOS` and `EOS` mark the beginning and end of a sequence, respectively.
- The `Lang` class will help manage the vocabulary, mapping words to indices and vice-versa.
- Text normalization is a standard step in NLP to simplify the vocabulary.

In [9]:
PAD_token = 0
SOS_token = 1
EOS_token = 2

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS"}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

def readLangs(lang1, lang2, reverse=False):
    lines = open('data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)
    return input_lang, output_lang, pairs

MAX_LENGTH = 10
eng_prefixes = (
    "i am ", "i m ", "he is", "he s ", "she is", "she s ",
    "you are", "you re ", "we are", "we re ", "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print(f"Read {len(pairs)} sentence pairs")
    pairs = filterPairs(pairs)
    print(f"Trimmed to {len(pairs)} sentence pairs")
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(f"{input_lang.name}: {input_lang.n_words}")
    print(f"{output_lang.name}: {output_lang.n_words}")
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))

Read 135842 sentence pairs
Trimmed to 10599 sentence pairs
Counting words...
Counted words:
fra: 4346
eng: 2804
['je suis loin d etre satisfait du resultat .', 'i am far from satisfied with the result .']


In [10]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long)

def tensorsFromPair(pair):
    src_tensor = tensorFromSentence(input_lang, pair[0])
    tgt_tensor = tensorFromSentence(output_lang, pair[1])
    decoder_input = torch.cat(
        [torch.tensor([SOS_token], dtype=torch.long), tgt_tensor[:-1]]
    )
    return src_tensor, decoder_input, tgt_tensor

def get_dataloader(batch_size, pairs):
    src_tensors = []
    tgt_inputs = []
    tgt_outputs = []

    for pair in pairs:
        src_tensor, tgt_input, tgt_output = tensorsFromPair(pair)
        src_tensors.append(src_tensor)
        tgt_inputs.append(tgt_input)
        tgt_outputs.append(tgt_output)

    src_padded = pad_sequence(src_tensors, batch_first=True, padding_value=PAD_token)
    tgt_in_padded = pad_sequence(tgt_inputs, batch_first=True, padding_value=PAD_token)
    tgt_out_padded = pad_sequence(tgt_outputs, batch_first=True, padding_value=PAD_token)

    dataset = TensorDataset(src_padded, tgt_in_padded, tgt_out_padded)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

<hr>

## Part B: Transformer Model

This includes the `PositionalEncoding` and the main `TransformerModel`.

<span style="color: orange; font-size: 28px; font-weight: bold">Task 1: Implement the Positional Encoding. (20 Marks)</span>

**Hint on Positional Encoding:**
- The Transformer does not have a built-in sense of sequence order. Positional encodings are added to the input embeddings to give the model information about the position of each token.
- We use sine and cosine functions of different frequencies.

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)              # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # 偶数位置: sin, 奇数位置: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 形状调整为 (max_len, 1, d_model) 方便与 (seq_len, batch, d_model) 相加
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)  # 不参与训练
    def forward(self, x):
        # x: (seq_len, batch, d_model)
        x = x + self.pe[:x.size(0)]
        return x

<span style="color: orange; font-size: 28px; font-weight: bold">Task 2: Complete the `forward` pass of the `TransformerModel`. (20 Marks)</span>
- The `forward` pass defines how data flows through the model.
- You need to handle embeddings, positional encoding, padding masks, and the decoder's autoregressive mask.

**Hint on the `forward` pass:**
- Apply embeddings and scale them.
- Add positional encodings.
- Create the target mask to prevent the decoder from looking at future tokens.
- Pass the data through the encoder and decoder.
- The final output should be a linear layer to get logits over the vocabulary.

In [12]:
class TransformerModel(nn.Module):
    def __init__(self, input_vocab_size, output_vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.model_type = 'Transformer'
        self.d_model = d_model
        self.pos_encoder = PositionalEncoding(d_model)
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(output_vocab_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=False)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=False)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)

        self.fc_out = nn.Linear(d_model, output_vocab_size)

    def forward(self, src, tgt, src_padding_mask=None, tgt_padding_mask=None, memory_key_padding_mask=None):
        # src, tgt: (batch, seq)
        # 1. 嵌入 + 缩放
        src_emb = self.encoder_embedding(src) * math.sqrt(self.d_model)   # (B,S,E)
        tgt_emb = self.decoder_embedding(tgt) * math.sqrt(self.d_model)   # (B,T,E)

        # 2. 维度置换 -> (seq_len, batch, d_model)
        src_emb = src_emb.permute(1, 0, 2)
        tgt_emb = tgt_emb.permute(1, 0, 2)

        # 3. 加位置编码
        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)

        # 4. 生成 decoder mask (防止看到未来)
        T = tgt_emb.size(0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T).to(tgt_emb.device)

        # 5. 编码 / 解码
        memory = self.encoder(src_emb, src_key_padding_mask=src_padding_mask)  # (S,B,E)
        out = self.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )  # (T,B,E)

        # 6. 映射到词表
        out = self.fc_out(out)  # (T,B,V)

        # 返回 (B,T,V)
        return out.permute(1, 0, 2)


def create_padding_mask(seq):
    return (seq == PAD_token)

In [13]:
input_vocab_size = input_lang.n_words
output_vocab_size = output_lang.n_words
d_model = 128
nhead = 4
num_encoder_layers = 2
num_decoder_layers = 2
dim_feedforward = 512
dropout = 0.1

model = TransformerModel(
    input_vocab_size, output_vocab_size, d_model, nhead,
    num_encoder_layers, num_decoder_layers, dim_feedforward, dropout
).to(device)

def init_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight)
model.apply(init_weights)

print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.')

/home/comp/cswjyu/anaconda3/envs/torchrec_py310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


The model has 2,202,612 trainable parameters.


<hr>

## Part C: Training and Evaluation
<span style="color: orange; font-size: 28px; font-weight: bold">Task 3: Implement the training loop and evaluation function. (20 Marks)</span>

**Hint for `train_epoch`:**
- Set the model to training mode.
- For each batch, create padding masks for source and target.
- Get the model output and compute the loss.
- Use `CrossEntropyLoss` and remember to set `ignore_index=PAD_token`.
- Backpropagate and update weights.

In [14]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for src, tgt_input, tgt_out in dataloader:
        src, tgt_input, tgt_out = src.to(device), tgt_input.to(device), tgt_out.to(device)

        src_padding_mask = create_padding_mask(src)          # (B,S)
        tgt_padding_mask = create_padding_mask(tgt_input)    # (B,T)

        optimizer.zero_grad()

        output = model(
            src,
            tgt_input,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )  # (B,T,V)

        # 交叉熵需要 (B*T, V) 与 (B*T)
        loss = criterion(output.reshape(-1, output.size(-1)), tgt_out.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def train(model, dataloader, n_epochs, learning_rate=0.0005):
    print("Training...")
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, dataloader, optimizer, criterion)
        print(f"Epoch {epoch}, Loss: {loss:.4f}")
    return model

In [15]:
# This cell will fail until you complete the functions above
batch_size = 32
train_dataloader = get_dataloader(batch_size, pairs)
model = train(model, train_dataloader, n_epochs=20)

Training...


/home/comp/cswjyu/anaconda3/envs/torchrec_py310/lib/python3.10/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch 1, Loss: 3.1749
Epoch 2, Loss: 1.8126
Epoch 3, Loss: 1.3033
Epoch 4, Loss: 0.9413
Epoch 5, Loss: 0.6785
Epoch 6, Loss: 0.4962
Epoch 7, Loss: 0.3710
Epoch 8, Loss: 0.2803
Epoch 9, Loss: 0.2184
Epoch 10, Loss: 0.1753
Epoch 11, Loss: 0.1437
Epoch 12, Loss: 0.1275
Epoch 13, Loss: 0.1117
Epoch 14, Loss: 0.1043
Epoch 15, Loss: 0.0997
Epoch 16, Loss: 0.0930
Epoch 17, Loss: 0.0875
Epoch 18, Loss: 0.0829
Epoch 19, Loss: 0.0765
Epoch 20, Loss: 0.0744


<span style="color: orange; font-size: 28px; font-weight: bold">Task 4: Implement the `evaluate` function for greedy decoding.(25 Marks)</span>
- This function translates a sentence by repeatedly feeding the model's own output back to itself.
- <b> Among the total 25 marks, 5 marks will be given based on the result of the random evaluation. </b>

**Hint for `evaluate`:**
- Set the model to evaluation mode.
- Start the decoding with the `SOS_token`.
- In a loop, get the model's prediction for the next token.
- Append the predicted token to the decoded sequence.
- Stop when `EOS_token` is predicted or `max_length` is reached.

In [16]:
def evaluate(model, sentence, input_lang, output_lang, max_length=MAX_LENGTH):
    model.eval()
    with torch.no_grad():
        src_tensor = tensorFromSentence(input_lang, sentence).unsqueeze(0).to(device)  # (1,S)
        src_padding_mask = create_padding_mask(src_tensor)

        ys = torch.tensor([[SOS_token]], dtype=torch.long, device=device)  # (1,1)

        for _ in range(max_length):
            tgt_padding_mask = create_padding_mask(ys)  # (1,len)
            out = model(
                src_tensor,
                ys,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )  # (1,len,V)
            next_token_logits = out[0, -1, :]
            next_token = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)  # (1,1)
            ys = torch.cat([ys, next_token], dim=1)
            if next_token.item() == EOS_token:
                break

        decoded_words = []
        for token in ys[0, 1:]:
            idx = token.item()
            if idx == EOS_token:
                break
            decoded_words.append(output_lang.index2word.get(idx, "<unk>"))
        return " ".join(decoded_words)

def evaluateRandomly(model, n=5):
    for _ in range(n):
        src_sentence, tgt_sentence = random.choice(pairs)
        predicted = evaluate(model, src_sentence, input_lang, output_lang)
        print(f"> {src_sentence}")
        print(f"= {tgt_sentence}")
        print(f"< {predicted}\n")

In [17]:
# Randomly output some translations, to examine the model performance
evaluateRandomly(model)

> je suis marie maintenant .
= i m married now .
< i m married now .

> elles sont mauvaises .
= they re bad .
< they re bad .

> nous allons bien maintenant .
= we re all right now .
< we re all right now .

> ton temps est ecoule .
= you re out of time .
< you re out of time .

> je suis en train d acheter un chiot .
= i m buying a puppy .
< i m buying a puppy .



<hr>

## Part D: Reflection
<span style="color: orange; font-size: 28px; font-weight: bold">Task 5: Answer the following questions with your own languages. (15 Marks)</span>
- 1. After implementing and training your model, reflect on its performance.
- 2. What are the strengths and weaknesses of this Transformer model compared to an RNN-based model for this task?
- 3. How might you improve the model's performance? (e.g., more data, larger model, different decoding strategy).
- <b> Write your thoughts in the markdown cell below. </b>

*Write your reflection here.*
- 1) Model performance
  - After training, the model produced exact matches on the sampled test sentences. It clearly captured frequent patterns like “be + adjective/state” and short, templated utterances.
  - It’s fluent and grammatical on short inputs but may struggle with longer dependencies, rare words, or out-of-distribution phrasing.
  - Outputs can be a bit templated, reflecting high-frequency patterns from the training data.

- 2) Transformer vs. RNN
  - Strengths (Transformer): parallelizable training/inference on sequences; self-attention models long-range dependencies well; often converges faster and learns richer representations.
  - Weaknesses (for small data): larger capacity and more hyperparameters can overfit; more data-hungry; needs careful tuning (e.g., positional encoding, dropout). RNNs can be more stable on tiny datasets.

- 3) Possible improvements
  - Data & vocab: use more training data; adopt subword units (BPE/WordPiece) to reduce OOV; relax filtering to include more diverse sentence structures.
  - Training: add label smoothing; tie input/output embeddings; use LR warmup + scheduler; gradient clipping; stronger regularization (dropout/weight decay); early stopping with best-checkpoint saving.
  - Model & decoding: modestly increase d_model/layers; use beam search with length penalty; consider pretrained embeddings; evaluate with BLEU/chrF on a held-out validation set to monitor generalization.